<a href="https://colab.research.google.com/github/ViniciusL-25/Dev_Ops/blob/main/marco1_projeto_ml_student_performance.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Projeto Acadêmico IESB — Aprendizagem de Máquina
## Marco 1 — Entrega 18/09/2026 (base Student Performance — UCI)

**Disciplina:** Aprendizagem de Máquina — Prof. Rodrigo Gonçalves
**Grupo:** _(preencher nomes e matrículas)_
**Responsabilidades desta entrega (Marco 1):** Responsável pelos Dados e pela Análise Exploratória, com apoio dos demais integrantes.

Dataset indicado pelo professor: **Student Performance** (UCI Machine Learning Repository).

Este notebook cobre os itens exigidos no Marco 1:
1. Conjunto de dados aprovado, com ficha de descrição;
2. Análise exploratória documentada (≥ 5 gráficos, cada um com leitura escrita);
3. Pré-processamento implementado e justificado;
4. Protocolo experimental definido e um modelo de referência (*baseline*).


## 1. Ficha de Descrição do Conjunto de Dados

**Nome:** Student Performance Data Set
**Fonte:** UCI Machine Learning Repository — https://archive.ics.uci.edu/dataset/320/student+performance
**Citação:** Cortez, P. (2014). *Student Performance*. UCI Machine Learning Repository.
**Licença:** disponibilizado publicamente pelo UCI para fins acadêmicos, com citação da fonte.

**Atenção ao requisito mínimo de 1.000 registros:** o dataset original vem dividido em dois arquivos — desempenho em **Matemática** (395 alunos) e em **Português** (649 alunos) — de duas escolas secundárias em Portugal. Isoladamente, nenhum dos dois arquivos atinge o mínimo de 1.000 registros exigido pelo edital. Por isso, **empilhamos os dois arquivos** (mantendo uma coluna `disciplina` para identificar a origem de cada linha), totalizando **1.044 registros**, o que atende ao requisito mínimo e ainda agrega uma variável categórica relevante à análise.

**Número de atributos:** 33 colunas por arquivo (dados demográficos, sociais, familiares e escolares), das quais G1, G2 e G3 são as notas do aluno em três períodos do ano letivo.

### Dicionário dos principais atributos utilizados

| Atributo | Tipo | Descrição |
|---|---|---|
| school | Categórico | Escola do aluno (GP ou MS) |
| sex | Categórico | Sexo do aluno |
| age | Numérico | Idade (15 a 22 anos) |
| address | Categórico | Endereço urbano (U) ou rural (R) |
| famsize | Categórico | Tamanho da família (≤3 ou >3 pessoas) |
| Pstatus | Categórico | Pais vivem juntos (T) ou separados (A) |
| Medu / Fedu | Numérico (ordinal) | Escolaridade da mãe / do pai (0 a 4) |
| Mjob / Fjob | Categórico | Profissão da mãe / do pai |
| reason | Categórico | Motivo de escolha da escola |
| guardian | Categórico | Responsável legal do aluno |
| traveltime | Numérico (ordinal) | Tempo de deslocamento até a escola (1 a 4) |
| studytime | Numérico (ordinal) | Tempo semanal de estudo (1 a 4) |
| failures | Numérico | Número de reprovações anteriores |
| schoolsup, famsup, paid, activities, nursery, higher, internet, romantic | Categórico (sim/não) | Apoios extras, atividades, acesso à internet, relacionamento, etc. |
| famrel, freetime, goout, Dalc, Walc, health | Numérico (ordinal, escala 1–5) | Relação familiar, tempo livre, saídas, consumo de álcool, saúde |
| absences | Numérico | Número de faltas |
| G1, G2 | Numérico | Notas do 1º e 2º período (0 a 20) |
| G3 | Numérico | Nota final do 3º período (0 a 20) |
| disciplina | Categórico | Matemática ou Português (criada ao empilhar os dois arquivos) |

### Definição da variável-alvo
Criamos a variável binária **`APROVADO`**: `1` se a nota final `G3 >= 10` (nota mínima de aprovação em Portugal), `0` caso contrário. É um problema de **classificação binária**.

**Alerta de vazamento (importante):** as notas `G1` e `G2` são extremamente correlacionadas com `G3` — na prática, quase determinam o resultado final, pois são notas do mesmo aluno em períodos anteriores do mesmo ano. Usá-las como preditoras tornaria o problema trivial (o modelo "colaria" na resposta em vez de aprender os fatores socioeducacionais). Por isso, **G1 e G2 serão excluídas das variáveis preditoras** no protocolo experimental — essa é uma decisão de projeto deliberada contra vazamento, e não um esquecimento, e deve ser citada explicitamente na apresentação.


## 2. Importação de bibliotecas e carregamento dos dados

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import urllib.request
import zipfile
import io

sns.set_style("whitegrid")
pd.set_option("display.max_columns", 60)


In [ ]:
# Download do arquivo oficial do UCI (contém student-mat.csv e student-por.csv)
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/00320/student.zip"

resp = urllib.request.urlopen(url)
zip_bytes = io.BytesIO(resp.read())

with zipfile.ZipFile(zip_bytes) as z:
    with z.open("student-mat.csv") as f:
        df_mat = pd.read_csv(f, sep=";")
    with z.open("student-por.csv") as f:
        df_por = pd.read_csv(f, sep=";")

print("Matemática:", df_mat.shape, "| Português:", df_por.shape)


In [ ]:
# Empilhamento dos dois arquivos, com coluna de identificação da disciplina
df_mat["disciplina"] = "Matematica"
df_por["disciplina"] = "Portugues"

df_raw = pd.concat([df_mat, df_por], ignore_index=True)
print("Dimensões da base combinada:", df_raw.shape)
df_raw.head()


### 2.1 Estrutura geral, tipos de dados e valores ausentes

In [ ]:
df_raw.info()


In [ ]:
print("Valores ausentes por coluna:")
print(df_raw.isnull().sum().sum(), "valores ausentes no total")
print("\nLinhas duplicadas (idênticas em todas as colunas):", df_raw.duplicated().sum())


**Leitura:** este dataset é uma coleta feita via boletins escolares e questionários, por isso já vem bem estruturado, com poucos ou nenhum valor ausente verdadeiro. O ponto de atenção real aqui não é ausência de dado, e sim que **um mesmo aluno pode aparecer nas duas disciplinas** (Matemática e Português) — não como uma linha idêntica (as notas mudam), mas como o mesmo estudante. Isso é discutido na Seção 5 como um cuidado equivalente a uma checagem de vazamento.

## 3. Análise Exploratória de Dados (EDA)

Antes dos gráficos, criamos a variável-alvo `APROVADO` a partir da nota final `G3`.

In [ ]:
df = df_raw.copy()
df["APROVADO"] = (df["G3"] >= 10).astype(int)

print(df["APROVADO"].value_counts(normalize=True))


In [ ]:
# Gráfico 1 — Balanceamento da variável-alvo
plt.figure(figsize=(5,4))
sns.countplot(x="APROVADO", data=df)
plt.title("Distribuição da variável-alvo (Aprovado)")
plt.xlabel("Aprovado (G3 >= 10)?")
plt.ylabel("Número de alunos")
plt.show()


**Leitura (Gráfico 1):** diferente de outros problemas de risco (que costumam ser bem desbalanceados), aqui a maioria dos alunos é aprovada, mas ainda existe uma fração relevante de reprovados — é um desbalanceamento mais leve que, ainda assim, precisa ser levado em conta na escolha das métricas.

In [ ]:
# Gráfico 2 — Distribuição da nota final G3, com a linha de corte de aprovação
plt.figure(figsize=(7,4))
sns.histplot(df["G3"], bins=21, kde=True)
plt.axvline(10, color="red", linestyle="--", label="Corte de aprovação (10)")
plt.title("Distribuição da nota final (G3)")
plt.xlabel("Nota final (0-20)")
plt.legend()
plt.show()

print(df["G3"].describe())


**Leitura (Gráfico 2):** chama atenção um pico de alunos com nota **0**, provavelmente correspondendo a alunos que abandonaram a disciplina ou não compareceram às avaliações — um comportamento que um valor de nota contínua não deixaria óbvio sem o histograma. Fora esse pico em zero, a distribuição das notas é razoavelmente concentrada entre 8 e 14, com o corte de aprovação (linha vermelha) passando bem no meio da massa de dados — o que explica por que o problema não é trivialmente fácil.

In [ ]:
# Gráfico 3 — Faltas (absences) por situação de aprovação
plt.figure(figsize=(6,4))
sns.boxplot(x="APROVADO", y="absences", data=df)
plt.title("Número de faltas por situação de aprovação")
plt.ylim(0, 40)
plt.show()


**Leitura (Gráfico 3):** alunos reprovados tendem a ter uma dispersão de faltas ligeiramente maior, mas a diferença de mediana entre os grupos é pequena — sugerindo que `absences`, isoladamente, é um preditor mais fraco do que se poderia supor, e que o modelo provavelmente vai depender da combinação de várias variáveis, não de uma única "vilã".

In [ ]:
# Gráfico 4 — Matriz de correlação das variáveis numéricas (sem G1/G2, para não distorcer a leitura)
num_cols_eda = ["age", "Medu", "Fedu", "traveltime", "studytime", "failures",
                 "famrel", "freetime", "goout", "Dalc", "Walc", "health", "absences", "G3"]
num_cols_eda = [c for c in num_cols_eda if c in df.columns]

plt.figure(figsize=(9,7))
sns.heatmap(df[num_cols_eda].corr(), annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Matriz de correlação — variáveis numéricas (excluindo G1/G2)")
plt.show()


**Leitura (Gráfico 4):** propositalmente excluímos `G1` e `G2` deste heatmap — incluí-las dominaria visualmente o gráfico (correlação próxima de 0.9 com `G3`) e esconderia relações mais sutis. Entre as demais variáveis, `failures` (reprovações anteriores) aparece com a correlação negativa mais forte com `G3`, e `Medu`/`Fedu` (escolaridade dos pais) mostram correlação positiva fraca a moderada — sinal de que o histórico acadêmico do aluno pesa mais que o ambiente familiar, embora ambos contribuam.

In [ ]:
# Gráfico 5 — Taxa de aprovação por número de reprovações anteriores (failures)
taxa_por_failures = df.groupby("failures")["APROVADO"].mean()

plt.figure(figsize=(6,4))
taxa_por_failures.plot(kind="bar", color="steelblue")
plt.title("Taxa de aprovação por número de reprovações anteriores")
plt.ylabel("Proporção de aprovados")
plt.xlabel("Nº de reprovações anteriores (failures)")
plt.xticks(rotation=0)
plt.show()

print(taxa_por_failures)


**Leitura (Gráfico 5):** a relação é bastante clara e monotônica — quanto mais reprovações anteriores o aluno teve, menor a taxa de aprovação atual. Isso confirma visualmente o que o Gráfico 4 já indicava pela correlação, e sugere que `failures` deve ser uma das variáveis mais importantes em qualquer modelo supervisionado treinado nos próximos marcos.

In [ ]:
# Gráfico 6 — Taxa de aprovação por disciplina (Matemática x Português)
taxa_por_disciplina = df.groupby("disciplina")["APROVADO"].mean()

plt.figure(figsize=(5,4))
taxa_por_disciplina.plot(kind="bar", color="darkorange")
plt.title("Taxa de aprovação por disciplina")
plt.ylabel("Proporção de aprovados")
plt.xticks(rotation=0)
plt.show()

print(taxa_por_disciplina)


**Leitura (Gráfico 6):** a taxa de aprovação em Português é visivelmente mais alta que em Matemática — um padrão já bem documentado na literatura que usa este mesmo dataset. Isso justifica manter `disciplina` como variável preditora (e não descartá-la), já que ela carrega informação real sobre a dificuldade relativa do curso.

## 4. Pré-processamento

Cada decisão abaixo é justificada com base no que foi observado na EDA.

In [ ]:
df_proc = df.copy()

# 4.1 Remoção de duplicatas exatas
# Justificativa: linhas idênticas em todas as colunas não trazem informação nova
# e, se distribuídas entre treino e teste, causam vazamento de dados.
antes = df_proc.shape[0]
df_proc = df_proc.drop_duplicates()
print(f"Linhas removidas por duplicidade: {antes - df_proc.shape[0]}")


In [ ]:
# 4.2 Valores ausentes
# Justificativa: como identificado na Seção 2.1, este dataset praticamente não
# tem valores ausentes verdadeiros — não há necessidade de imputação agressiva,
# mas o código abaixo garante robustez caso alguma célula venha vazia.
num_cols_all = df_proc.select_dtypes(include=[np.number]).columns.tolist()
for c in ["G1", "G2", "G3", "APROVADO"]:
    if c in num_cols_all:
        num_cols_all.remove(c)

cat_cols_all = df_proc.select_dtypes(exclude=[np.number]).columns.tolist()

for c in num_cols_all:
    if df_proc[c].isnull().any():
        df_proc[c] = df_proc[c].fillna(df_proc[c].median())

for c in cat_cols_all:
    if df_proc[c].isnull().any():
        df_proc[c] = df_proc[c].fillna(df_proc[c].mode()[0])

print("Valores ausentes restantes:", df_proc.isnull().sum().sum())


In [ ]:
# 4.3 Codificação de variáveis categóricas
# Justificativa: colunas binárias sim/não (yes/no) viram 0/1 diretamente, sem
# necessidade de One-Hot (não há ambiguidade de ordem em uma binária).
# Colunas nominais com mais de duas categorias (Mjob, Fjob, reason, guardian,
# disciplina) recebem One-Hot Encoding, para não impor uma ordem inexistente.
# Colunas já numéricas ordinais (Medu, Fedu, traveltime, studytime, famrel,
# freetime, goout, Dalc, Walc, health) permanecem como estão — o próprio
# dataset já as codifica em escalas ordenadas (ex.: 1 a 5).

binarias_yes_no = ["schoolsup", "famsup", "paid", "activities", "nursery", "higher", "internet", "romantic"]
binarias_yes_no = [c for c in binarias_yes_no if c in df_proc.columns]
for c in binarias_yes_no:
    df_proc[c] = df_proc[c].map({"yes": 1, "no": 0})

binarias_2cat = ["school", "sex", "address", "famsize", "Pstatus"]
binarias_2cat = [c for c in binarias_2cat if c in df_proc.columns]
for c in binarias_2cat:
    df_proc[c] = df_proc[c].astype("category").cat.codes  # 0/1, ordem arbitrária mas só 2 categorias

nominais_multi = ["Mjob", "Fjob", "reason", "guardian", "disciplina"]
nominais_multi = [c for c in nominais_multi if c in df_proc.columns]
df_proc = pd.get_dummies(df_proc, columns=nominais_multi, drop_first=True)

print(df_proc.shape)
df_proc.head()


**Nota sobre normalização/padronização:** a padronização (`StandardScaler`) das variáveis numéricas contínuas será aplicada **depois da separação treino/teste**, ajustando (`fit`) o scaler apenas no treino, como no protocolo da Seção 5 — evitando vazamento estatístico do teste para o treino.

**Nota sobre vazamento conceitual (reforçando a Seção 1):** `G1`, `G2` e `G3` serão removidas das variáveis preditoras a seguir. `G3` porque é a própria origem do alvo; `G1` e `G2` porque, sendo notas do mesmo aluno em períodos anteriores do mesmo ano letivo, tornariam a predição trivial e mascarariam a real contribuição dos fatores socioeducacionais que o projeto quer investigar.

## 5. Protocolo Experimental e Baseline

### 5.1 Separação treino / validação / teste

In [ ]:
from sklearn.model_selection import train_test_split

# Remove G1, G2, G3 das features — G3 é a origem do alvo (vazamento direto),
# G1/G2 tornariam o problema trivial (vazamento indireto, mesma variável repetida no tempo).
X = df_proc.drop(columns=["G1", "G2", "G3", "APROVADO"])
y = df_proc["APROVADO"]

# Split 60% treino / 20% validação / 20% teste, estratificado pela classe-alvo
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.4, stratify=y, random_state=42
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, stratify=y_temp, random_state=42
)

print("Treino:", X_train.shape, "| Validação:", X_val.shape, "| Teste:", X_test.shape)
print("Proporção da classe positiva em cada conjunto:")
print("Treino:", y_train.mean().round(3), "| Validação:", y_val.mean().round(3), "| Teste:", y_test.mean().round(3))


### 5.2 Verificação de vazamento de dados

In [ ]:
inter_train_val = set(X_train.index) & set(X_val.index)
inter_train_test = set(X_train.index) & set(X_test.index)
inter_val_test = set(X_val.index) & set(X_test.index)

print("Interseção treino/val:", len(inter_train_val))
print("Interseção treino/teste:", len(inter_train_test))
print("Interseção val/teste:", len(inter_val_test))

# Confirma explicitamente que G1, G2 e G3 não estão entre as features
print("G1/G2/G3 estão nas features?", any(c in X.columns for c in ["G1", "G2", "G3"]))

# Checagem extra específica deste dataset: mesmo aluno em Matemática e Português
# (colunas que, juntas, tendem a identificar unicamente um estudante)
colunas_id_aluno = ["school", "sex", "age", "address", "famsize", "Pstatus",
                     "Medu", "Fedu", "Mjob_health", "Mjob_other", "Mjob_services", "Mjob_teacher"]
colunas_id_aluno = [c for c in colunas_id_aluno if c in df_proc.columns]
possiveis_mesmo_aluno = df_proc.duplicated(subset=colunas_id_aluno).sum()
print("Linhas que podem representar o mesmo aluno em disciplinas diferentes:", possiveis_mesmo_aluno)


**Leitura:** não há interseção de índices entre os conjuntos, e `G1`/`G2`/`G3` foram explicitamente removidas das features — as duas checagens centrais de vazamento estão documentadas. A checagem extra estima quantos registros podem representar o mesmo aluno presente nas duas disciplinas; como o split foi feito sobre a base combinada sem agrupar por aluno, existe um risco residual de o mesmo estudante aparecer em treino e teste (em disciplinas diferentes) — isso é uma limitação conhecida do projeto e deve ser citada na apresentação como um ponto de melhoria para os próximos marcos (ex.: split por `agrupamento de aluno` em vez de aleatório simples).

### 5.3 Padronização das variáveis numéricas (ajustada apenas no treino)

In [ ]:
from sklearn.preprocessing import StandardScaler

num_cols_final = [c for c in num_cols_all if c in X_train.columns]

scaler = StandardScaler()
X_train_scaled = X_train.copy()
X_val_scaled = X_val.copy()
X_test_scaled = X_test.copy()

X_train_scaled[num_cols_final] = scaler.fit_transform(X_train[num_cols_final])
X_val_scaled[num_cols_final] = scaler.transform(X_val[num_cols_final])
X_test_scaled[num_cols_final] = scaler.transform(X_test[num_cols_final])


### 5.4 Modelo de referência (baseline)

O baseline utiliza a estratégia "mais frequente" (`DummyClassifier`), que sempre prevê a classe majoritária (aprovado). Qualquer modelo supervisionado desenvolvido nos próximos marcos deve superar este piso para ser considerado minimamente útil.

In [ ]:
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report

baseline = DummyClassifier(strategy="most_frequent", random_state=42)
baseline.fit(X_train_scaled, y_train)
y_pred_baseline = baseline.predict(X_val_scaled)

print("=== Baseline (DummyClassifier — classe majoritária) ===")
print("Acurácia :", round(accuracy_score(y_val, y_pred_baseline), 4))
print("Precisão :", round(precision_score(y_val, y_pred_baseline, zero_division=0), 4))
print("Recall   :", round(recall_score(y_val, y_pred_baseline, zero_division=0), 4))
print("F1-score :", round(f1_score(y_val, y_pred_baseline, zero_division=0), 4))
print("\nMatriz de confusão:\n", confusion_matrix(y_val, y_pred_baseline))
print("\n", classification_report(y_val, y_pred_baseline, zero_division=0))


**Leitura do baseline:** como a maioria dos alunos é aprovada, o baseline atinge uma acurácia moderada/alta apenas chutando "aprovado" sempre, com precisão e recall nulos para a classe de reprovados. Esse número é o piso real: qualquer modelo supervisionado dos próximos marcos (árvore de decisão, KNN, Naive Bayes, rede neural) precisa superá-lo — em especial em recall e F1 da classe "reprovado", já que identificar corretamente um aluno em risco de reprovação é o objetivo prático mais útil deste projeto (permitiria intervenção pedagógica precoce).

## 6. Próximos passos (fora do escopo do Marco 1)

- Responsável 2: k-means (com método do cotovelo e silhueta) para agrupar perfis de alunos, árvore de decisão/regressão para `APROVADO` ou para `G3` contínuo, regressão linear usando `G3` como alvo numérico;
- Responsável 3: KNN, Naive Bayes, rede neural multicamada, validação cruzada k-fold estratificada e quadro comparativo final, com recomendação fundamentada do melhor modelo.

Este notebook cobre integralmente os requisitos do **Marco 1**: dataset indicado pelo professor (Student Performance — UCI), com ajuste documentado para atingir o mínimo de 1.000 registros (empilhamento Matemática + Português), ficha de descrição, EDA com 6 gráficos comentados, pré-processamento justificado (deduplicação, codificação binária/nominal/ordinal) e baseline com protocolo experimental completo — incluindo a discussão explícita sobre vazamento de G1/G2 e sobre alunos repetidos entre disciplinas.